### meteo.data Cartographie des ARCHIVES de VIGILANCES Météo-France de l'hexagone
- data : https://meteo.data.gouv.fr/datasets/69cb8c3efb376113fa42881a
- Auteur: https://github.com/loicduffar (et gemini PRO)

A ce stade la type de vigilance n'est pas précisé (ce script a servi à cartographier les vigilances CANICULE exceptionnelles de l'ensemble de l'hexagone en juin et jullet 2026)

NB: nécessite le module s3fs (il est installé dans mon environnement 311_opencv)

In [ ]:
##########################################
# Auteur : https://github.com/loicduffar
# juillet 2026
##########################################

import json
import os
import math
import urllib.request
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.patches as mpatches
from PIL import Image

import ssl
import certifi
import json
import pandas as pd

# Contournement local de  import s3fs: certains environnements Windows produisent l'erreur
# [ASN1: NOT_ENOUGH_DATA] lors du chargement du magasin de certificats système.
_ssl_create_default_context = ssl.create_default_context

def _create_default_context_with_certifi(*args, **kwargs):
    if not any(key in kwargs for key in ('cafile', 'capath', 'cadata')):
        kwargs['cafile'] = certifi.where()
    return _ssl_create_default_context(*args, **kwargs)

ssl.create_default_context = _create_default_context_with_certifi

import s3fs ######################  doit se situer après le contournement SSL pour éviter les erreurs de certificat sur certains environnements Windows.

# ==========================================
# 0. CONFIGURATION DES CHEMINS
# ==========================================
dossier_sortie = r"X:\1-COMMUN\DIS\Documentation\Hydrologie\Documentation externe\Climat France\Météo-France\meteo.data\Vigilance météo\out"
mois_debut= "2026-06"
mois_fin= "2026-07"

# ==========================================
# 1. TÉLÉCHARGEMENT & EXTRACTION DES DONNÉES
# ==========================================
os.makedirs(dossier_sortie, exist_ok=True)
print("Connexion au serveur S3 et recherche des fichiers...")
fs = s3fs.S3FileSystem(anon=True, client_kwargs={'endpoint_url': 'https://object.files.data.gouv.fr'})

# On cible juin et juillet 2026
# motifs = [
#     "meteofrance/data/vigilance/metropole/2026/06/*/*/CDP_CARTE_EXTERNE.json",
#     "meteofrance/data/vigilance/metropole/2026/07/*/*/CDP_CARTE_EXTERNE.json"
# ]

motifs = [f"meteofrance/data/vigilance/metropole/{mois}/*/*/CDP_CARTE_EXTERNE.json" for mois in pd.date_range(mois_debut, mois_fin, freq='MS').strftime('%Y-%m')]

fichiers_trouves = []
for motif in motifs:
    fichiers_trouves.extend(fs.glob(motif))

print(f"{len(fichiers_trouves)} fichiers trouvés. Extraction en cours...")

donnees_brutes = []

for chemin_fichier in fichiers_trouves:
    # Extraction de la date depuis le chemin : .../2026/07/14/...
    parties = chemin_fichier.split('/')
    annee, mois, jour = parties[4], parties[5], parties[6]
    date_jour = f"{annee}-{mois}-{jour}"
    
    try:
        with fs.open(chemin_fichier, 'rb') as f:
            data_json = json.load(f)
            
            # Navigation dans la structure du JSON
            domaines = data_json.get('product', {}).get('periods', [])[0].get('timelaps', {}).get('domain_ids', [])
            
            for dom in domaines:
                code_dept = str(dom.get('domain_id'))
                alerte = dom.get('max_color_id', 1)
                
                # On ne garde que les départements métropolitains (ex: '01', '75', '2A')
                # On ignore les codes spécifiques (FRA, 0610, etc.)
                if len(code_dept) == 2:
                    donnees_brutes.append({
                        'date': date_jour,
                        'code': code_dept,
                        'alerte': alerte
                    })
    except Exception as e:
        # Fichier corrompu ou structure inattendue
        continue

# Agrégation : On veut l'alerte MAXIMALE par département et par jour
df = pd.DataFrame(donnees_brutes)
df_journalier = df.groupby(['date', 'code'])['alerte'].max().reset_index()

# ==========================================
# 2. PRÉPARATION DU FOND DE CARTE
# ==========================================
print("Téléchargement du fond de carte...")
url_geojson = "https://france-geojson.gregoiredavid.fr/repo/departements.geojson"
# gdf_france= gpd.read_file(url_geojson) # provoque un erreur SSL sur certains environnements Windows corrigée par le bloc ci-dessous

# Téléchargement robuste avec certifi, puis lecture locale pour éviter les erreurs SSL de gpd.read_file(URL).
geojson_local = os.path.join(dossier_sortie, "departements.geojson")
ssl_context = ssl.create_default_context(cafile=certifi.where())
with urllib.request.urlopen(url_geojson, context=ssl_context, timeout=60) as response:
    contenu_geojson = response.read()
with open(geojson_local, "wb") as f_geo:
    f_geo.write(contenu_geojson)
gdf_france = gpd.read_file(geojson_local)

# Couleurs officielles Météo-France (1: Vert, 2: Jaune, 3: Orange, 4: Rouge)
cmap = ListedColormap(['#31a354', '#fed976', '#fd8d3c', '#e31a1c'])
norm = BoundaryNorm([0.5, 1.5, 2.5, 3.5, 4.5], cmap.N)
legendes = [
    mpatches.Patch(color='#e31a1c', label='Vigilance Absolue'),
    mpatches.Patch(color='#fd8d3c', label='Soyez très Vigilant'),
    mpatches.Patch(color='#fed976', label='Soyez attentif'),
    mpatches.Patch(color='#31a354', label='Pas de vigilance'),
]

# Liste des jours triés
jours = sorted(df_journalier['date'].unique())

# ==========================================
# 3. GRAND SUBPLOTS (VUE GLOBALE)
# ==========================================
print("Génération de la planche Subplots complète...")
# Calcul automatique de la grille (ex: 7x7 ou 6x8)
n_jours = len(jours)
n_cols = 7
n_rows = math.ceil(n_jours / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*3, n_rows*3))
fig.suptitle("Évolution des Vigilances Météo Maximales - Été 2026", fontsize=20, fontweight='bold', y=0.98)
axes = axes.flatten()

for i, jour in enumerate(jours):
    ax = axes[i]
    df_jour = df_journalier[df_journalier['date'] == jour]
    carte_jour = gdf_france.merge(df_jour, on='code', how='left')
    carte_jour['alerte'] = carte_jour['alerte'].fillna(1) # Vert par défaut si manquant
    
    carte_jour.plot(column='alerte', ax=ax, cmap=cmap, norm=norm, linewidth=0.3, edgecolor='black')
    ax.set_title(jour, fontsize=10)
    ax.axis('off')

# Masquer les subplots vides à la fin
for i in range(n_jours, len(axes)):
    axes[i].axis('off')

plt.tight_layout()
chemin_subplots = os.path.join(dossier_sortie, "planche_recapitulative_ete2026.png")
plt.savefig(chemin_subplots, dpi=200, bbox_inches='tight')
plt.close()

# ==========================================
# 4. CARTES INDIVIDUELLES ET CRÉATION DU GIF
# ==========================================
print(f"Génération des {n_jours} cartes individuelles...")
chemins_images = []

for jour in jours:
    fig_indiv, ax_indiv = plt.subplots(figsize=(8, 8))
    
    df_jour = df_journalier[df_journalier['date'] == jour]
    carte_jour = gdf_france.merge(df_jour, on='code', how='left')
    carte_jour['alerte'] = carte_jour['alerte'].fillna(1)
    
    carte_jour.plot(column='alerte', ax=ax_indiv, cmap=cmap, norm=norm, linewidth=0.5, edgecolor='black')
    
    ax_indiv.set_title(f"Vigilance Maximale du {jour}", fontsize=16)
    ax_indiv.axis('off')
    ax_indiv.legend(handles=legendes, loc='lower left')
    
    # Nom du fichier pour ce jour précis
    fichier_png = os.path.join(dossier_sortie, f"vigilance_{jour}.png")
    plt.savefig(fichier_png, dpi=150, bbox_inches='tight')
    chemins_images.append(fichier_png)
    plt.close(fig_indiv)

print("Création du GIF animé...")
# Création du GIF à partir de la liste des PNGs générés
images_pour_gif = [Image.open(img) for img in chemins_images]
chemin_gif = os.path.join(dossier_sortie, "evolution_vigilance_ete2026.gif")

# Sauvegarde du GIF (duration = ms par image)
images_pour_gif[0].save(
    chemin_gif, 
    save_all=True, 
    append_images=images_pour_gif[1:], 
    duration=600, 
    loop=0
)

print(f"\n--- TERMINÉ ! ---")
print(f"Planche complète sauvegardée : {chemin_subplots}")
print(f"GIF dynamique sauvegardé : {chemin_gif}")
print(f"Toutes les données sont dans : {dossier_sortie}")

Génération de la planche Subplots complète...
Génération des 44 cartes individuelles...
Création du GIF animé...

--- TERMINÉ ! ---
Planche complète sauvegardée : X:\1-COMMUN\DIS\Documentation\Hydrologie\Documentation externe\Climat France\Météo-France\meteo.data\Vigilance météo\out\planche_recapitulative_ete2026.png
GIF dynamique sauvegardé : X:\1-COMMUN\DIS\Documentation\Hydrologie\Documentation externe\Climat France\Météo-France\meteo.data\Vigilance météo\out\evolution_vigilance_ete2026.gif
Toutes les données sont dans : X:\1-COMMUN\DIS\Documentation\Hydrologie\Documentation externe\Climat France\Météo-France\meteo.data\Vigilance météo\out
